In [ ]:

import cv2
import numpy as np
import os

INPUT_DIR = '/home/user/output/dataset'
OUTPUT_DIR = '/home/user/output/final_annotated'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def preprocess(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=2)
    return closed

def find_paper_roi(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    _, th = cv2.threshold(blur, 190, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (20, 20))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=3)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < image.shape[0] * image.shape[1] * 0.15:
        return None
    x, y, w, h = cv2.boundingRect(c)
    return (x, y, w, h)

def classify(contour, shape_hint):
    area = cv2.contourArea(contour)
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0
    peri = cv2.arcLength(contour, True)
    circularity = (4 * np.pi * area / (peri * peri)) if peri > 0 else 0
    x, y, w, h = cv2.boundingRect(contour)
    extent = area / (w * h) if w * h > 0 else 0

    if shape_hint == 'circular':
        intact = solidity >= 0.96 and circularity >= 0.80
    else:
        intact = solidity >= 0.95 and extent >= 0.70

    label = 'Intact Biscuit' if intact else 'Broken Biscuit'
    color = (34, 177, 76) if intact else (0, 60, 220)
    return label, color

results_summary = []

for fname in sorted(os.listdir(INPUT_DIR)):
    if not fname.lower().endswith('.jpg'):
        continue

    image = cv2.imread(os.path.join(INPUT_DIR, fname))
    if image is None:
        continue

    h_img, w_img = image.shape[:2]
    shape_hint = 'circular' if 'circular' in fname.lower() else 'rectangular'

    paper = find_paper_roi(image)
    roi = image.copy()
    ox, oy = 0, 0
    if paper is not None:
        px, py, pw, ph = paper
        pad = 8
        px, py = max(0, px+pad), max(0, py+pad)
        pw = min(w_img - px, pw - 2*pad)
        ph = min(h_img - py, ph - 2*pad)
        roi = image[py:py+ph, px:px+pw].copy()
        ox, oy = px, py

    binary = preprocess(roi)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    roi_area = roi.shape[0] * roi.shape[1]
    min_area = max(800, int(roi_area * 0.003))
    valid = [c for c in contours if cv2.contourArea(c) >= min_area]

    annotated = image.copy()
    intact_n, broken_n = 0, 0
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.65
    thickness = 2

    for cnt in valid:
        label, color = classify(cnt, shape_hint)
        x, y, w, h = cv2.boundingRect(cnt)
        x += ox; y += oy
        cv2.rectangle(annotated, (x, y), (x+w, y+h), color, 3)
        (tw, th_), _ = cv2.getTextSize(label, font, font_scale, thickness)
        ty = max(th_ + 6, y - 6)
        cv2.rectangle(annotated, (x, ty - th_ - 4), (x + tw + 6, ty + 2), color, -1)
        cv2.putText(annotated, label, (x + 3, ty - 2), font, font_scale, (255, 255, 255), thickness)
        if 'Intact' in label:
            intact_n += 1
        else:
            broken_n += 1

    summary = f"Detected: {intact_n+broken_n}  |  Intact: {intact_n}  |  Broken: {broken_n}"
    cv2.rectangle(annotated, (0, 0), (w_img, 42), (30, 30, 30), -1)
    cv2.putText(annotated, summary, (10, 28), font, 0.75, (255, 255, 255), 2)

    out_path = os.path.join(OUTPUT_DIR, f'annotated_{fname}')
    cv2.imwrite(out_path, annotated, [cv2.IMWRITE_JPEG_QUALITY, 95])
    results_summary.append((fname, intact_n + broken_n, intact_n, broken_n))
    print(f" {fname}: {intact_n+broken_n} total | {intact_n} intact | {broken_n} broken")

print(f"\n All {len(results_summary)} images processed and saved to {OUTPUT_DIR}")